# Data collection for the year 2016

In [6]:
import cocopp
dsl = cocopp.load("bbob/2016/*")

In [7]:
import numpy as np

dd = dsl.dictByDimFunc()     # your grouped datasets
t = 1e-8                     # choose the target precision

best_by_df = {}              # (dim, fid) -> (best_alg, best_ert)

for dim in sorted(dd.keys()): 
    for fid in sorted(dd[dim].keys()):
        rows = []
        for ds in dd[dim][fid]:                 # each ds = one algorithm
            ert = float(ds.detERT([t])[0])      # ERT in #evals at target t
            rows.append((ds.algId, ert))  
        # ignore INF (not reached) when picking best
        finite = [(a, e) for (a, e) in rows if np.isfinite(e)] 
    
        if finite:
            best_alg, best_ert = min(finite, key=lambda x: x[1]) 
        else:
            best_alg, best_ert = None, np.inf
        best_by_df[(dim, fid)] = (best_alg, best_ert) 
        print(f"dim={dim:>2}, F{fid:>2} -> {best_alg}  (ERT={best_ert:.3g} @ {t})")


dim= 2, F 1 -> PSAaLmD-CMA-ES_Nishida  (ERT=390 @ 1e-08)
dim= 2, F 2 -> PSAaLmC-CMA-ES_Nishida  (ERT=2.2e+03 @ 1e-08)
dim= 2, F 3 -> PSAaSmD-CMA-ES_Nishida  (ERT=4.68e+03 @ 1e-08)
dim= 2, F 4 -> PSAaSmD-CMA-ES_Nishida  (ERT=2.71e+04 @ 1e-08)
dim= 2, F 5 -> PSAaLmD-CMA-ES_Nishida  (ERT=75.3 @ 1e-08)
dim= 2, F 6 -> PSAaLmD-CMA-ES_Nishida  (ERT=955 @ 1e-08)
dim= 2, F 7 -> PSAaLmD-CMA-ES_Nishida  (ERT=264 @ 1e-08)
dim= 2, F 8 -> PSAaLmC-CMA-ES_Nishida  (ERT=3.08e+03 @ 1e-08)
dim= 2, F 9 -> PSAaLmC-CMA-ES_Nishida  (ERT=2.74e+03 @ 1e-08)
dim= 2, F10 -> PSAaLmD-CMA-ES_Nishida  (ERT=1.95e+03 @ 1e-08)
dim= 2, F11 -> PSAaLmD-CMA-ES_Nishida  (ERT=1.51e+03 @ 1e-08)
dim= 2, F12 -> PSAaLmD-CMA-ES_Nishida  (ERT=1.18e+04 @ 1e-08)
dim= 2, F13 -> PSAaLmD-CMA-ES_Nishida  (ERT=1.39e+03 @ 1e-08)
dim= 2, F14 -> PSAaLmD-CMA-ES_Nishida  (ERT=805 @ 1e-08)
dim= 2, F15 -> PSAaLmD-CMA-ES_Nishida  (ERT=2.68e+03 @ 1e-08)
dim= 2, F16 -> PSAaSmD-CMA-ES_Nishida  (ERT=5.09e+03 @ 1e-08)
dim= 2, F17 -> PSAaLmD-CMA-ES_Nis

In [10]:
from collections import Counter, defaultdict

In [11]:
# Build a frequency counter: how many (dim,fid) each algo wins
win_counter = Counter(
    alg for (alg, ert) in best_by_df.values()
    if alg is not None and np.isfinite(ert)
)

# If you want a plain dict:
wins_dict = dict(win_counter)

# (Optional) pretty print, most wins first
for alg, count in win_counter.most_common():
    print(f"{alg}: {count}")

PSAaSmC-CMA-ES_Nishida: 32
PSAaSmD-CMA-ES_Nishida: 28
PSAaLmD-CMA-ES_Nishida: 27
PSAaLmC-CMA-ES_Nishida: 12


In [12]:
"""
Given best_by_df: {(dim, fid): (alg, ert)},
return {dim: algo_with_most_(fid)_wins_in_that_dim}.
Tie-break: lower total ERT across that dim, then alphabetical.
    """
wins = defaultdict(Counter)                    # dim -> Counter({alg: count})
ert_sums = defaultdict(lambda: defaultdict(float))  # dim -> {alg: total_ert}

for (dim, fid), (alg, ert) in best_by_df.items():
    if alg is None or not np.isfinite(ert):
        continue
    wins[dim][alg] += 1
    ert_sums[dim][alg] += float(ert)

result = {}
for dim, counter in wins.items():
    max_wins = max(counter.values())
    candidates = [a for a, c in counter.items() if c == max_wins]
    best = min(candidates, key=lambda a: (ert_sums[dim][a], a))  # tie-breaks
    result[dim] = best
result


{2: 'PSAaLmD-CMA-ES_Nishida',
 3: 'PSAaSmD-CMA-ES_Nishida',
 5: 'PSAaSmD-CMA-ES_Nishida',
 10: 'PSAaSmC-CMA-ES_Nishida',
 20: 'PSAaSmC-CMA-ES_Nishida'}

In [13]:
import numpy as np
import pandas as pd

# Make sure 'dd' already exists
# (if not, run: dsl = cocopp.load('path/to/your/ppdata'); dd = dsl.dictByDimFunc())

targets = [1e-1, 1e-2, 1e-3, 1e-5, 1e-8]
rows = []  # reset before starting the full loop

for dim in sorted(dd.keys()):                      # e.g. [2, 3, 5, 10, 20, 40]
    for fid in sorted(dd[dim].keys()):
        for t in targets:
            algo_erts = []
            for ds in dd[dim][fid]:                # each algorithm
                ert = float(ds.detERT([t])[0])
                algo_erts.append((ds.algId, ert))
            
            finite = [(a, e) for (a, e) in algo_erts if np.isfinite(e)]

            if finite:
                best_alg, best_ert = min(finite, key=lambda x: x[1])
            else:
                best_alg, best_ert = None, np.inf

            rows.append({
                "dimension": dim,
                "function_id": fid,
                "target": t,
                "best_algorithm": best_alg,
                "best_ERT": best_ert
            })

# Build DataFrame
df_best = pd.DataFrame(rows)
df_best = df_best.sort_values(by=["dimension", "function_id", "target"]).reset_index(drop=True)

# Confirm dimensions included
print("✅ Unique dimensions in table:", df_best["dimension"].unique())
print(df_best.head(15))


✅ Unique dimensions in table: [ 2  3  5 10 20]
    dimension  function_id        target          best_algorithm     best_ERT
0           2            1  1.000000e-08  PSAaLmD-CMA-ES_Nishida   389.733333
1           2            1  1.000000e-05  PSAaLmD-CMA-ES_Nishida   229.733333
2           2            1  1.000000e-03  PSAaLmD-CMA-ES_Nishida   153.733333
3           2            1  1.000000e-02  PSAaLmD-CMA-ES_Nishida    93.933333
4           2            1  1.000000e-01  PSAaLmD-CMA-ES_Nishida    54.866667
5           2            2  1.000000e-08  PSAaLmC-CMA-ES_Nishida  2195.600000
6           2            2  1.000000e-05  PSAaLmC-CMA-ES_Nishida  1653.400000
7           2            2  1.000000e-03  PSAaLmC-CMA-ES_Nishida  1343.266667
8           2            2  1.000000e-02  PSAaSmC-CMA-ES_Nishida  1195.800000
9           2            2  1.000000e-01  PSAaSmC-CMA-ES_Nishida   924.733333
10          2            3  1.000000e-08  PSAaSmD-CMA-ES_Nishida  4680.400000
11          2    

In [14]:
import os
os.makedirs("results", exist_ok=True)

df_best.to_csv("results/best_algos_2016.csv", index=False)
